# 15 - Task Diversity: Generate COTs

Generate COTs for SVAMP, ARC-Challenge, and HellaSwag using PRIMARY_MODEL.
Tests whether legibility varies by reasoning type.

In [1]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Already up to date.


In [ ]:
import json
import re
from tqdm import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from lib.data import (
    load_svamp, load_arc_challenge, load_hellaswag,
    extract_predicted_answer, extract_predicted_answer_mc,
)
from lib.prompts import build_cot_messages, build_no_cot_messages

# Load all task datasets
tasks = {
    "svamp": {
        "data": load_svamp(max_examples=300),
        "extractor": extract_predicted_answer,
        "cot_system": "Solve step by step. After your reasoning, write 'The answer is: ' followed by the numeric answer.",
    },
    "arc_challenge": {
        "data": load_arc_challenge(max_examples=500),
        "extractor": extract_predicted_answer_mc,
        "cot_system": "Think step by step. After your reasoning, write 'The answer is: ' followed by the letter (A, B, C, D, or E).",
    },
    "hellaswag": {
        "data": load_hellaswag(max_examples=500),
        "extractor": extract_predicted_answer_mc,
        "cot_system": "Think step by step about which continuation makes most sense. After your reasoning, write 'The answer is: ' followed by the letter (A, B, C, or D).",
    },
}

for name, task in tasks.items():
    print(f"{name}: {len(task['data'])} examples")

In [ ]:
print(f"Loading {PRIMARY_MODEL}...")
llm = LLM(model=PRIMARY_MODEL, dtype="bfloat16", max_model_len=4096)
tokenizer = AutoTokenizer.from_pretrained(PRIMARY_MODEL)
sampling_cot = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_COT_TOKENS)
sampling_short = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_ANSWER_TOKENS)

In [ ]:
import gc, torch

for task_name, task_info in tasks.items():
    data = task_info["data"]
    extractor = task_info["extractor"]
    system_prompt = task_info["cot_system"]

    # COT cache directory
    task_cot_dir = CACHE_DIR / f"cots_{task_name}"
    task_cot_dir.mkdir(parents=True, exist_ok=True)

    done_ids = {int(p.stem) for p in task_cot_dir.glob("*.json")}
    todo = [ex for ex in data if ex["problem_id"] not in done_ids]
    print(f"\n=== {task_name} ===")
    print(f"COTs: {len(done_ids)} done, {len(todo)} remaining")

    # Generate COTs
    for i in tqdm(range(0, len(todo), CHUNK_SIZE), desc=f"COTs ({task_name})"):
        chunk = todo[i:i + CHUNK_SIZE]
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "system", "content": system_prompt},
                 {"role": "user", "content": ex["question"]}],
                tokenize=False, add_generation_prompt=True,
                enable_thinking=True
            ) for ex in chunk
        ]
        outputs = llm.generate(prompts, sampling_cot)

        for ex, output in zip(chunk, outputs):
            full_response = output.outputs[0].text
            think_match = re.search(r"<think>(.*?)</think>", full_response, re.DOTALL)
            if think_match:
                cot_text = think_match.group(1).strip()
                answer_portion = full_response[think_match.end():].strip()
            else:
                parts = re.split(r"The answer is:?", full_response, maxsplit=1)
                cot_text = parts[0].strip()
                answer_portion = parts[1].strip() if len(parts) > 1 else full_response

            predicted = extractor(answer_portion)
            if predicted is None:
                predicted = extractor(full_response)

            result = {
                "problem_id": ex["problem_id"],
                "question": ex["question"],
                "gold_answer": ex["gold_answer"],
                "cot_text": cot_text,
                "predicted_answer": predicted,
                "full_response": full_response,
                "task": task_name,
            }
            (task_cot_dir / f"{ex['problem_id']}.json").write_text(json.dumps(result))

    # No-COT baseline
    nc_cond = f"no_cot_{task_name}"
    nc_done = {int(p.stem.split("_")[-1]) for p in PREFILL_CACHE.glob(f"{nc_cond}_*.json")}
    nc_todo = [ex for ex in data if ex["problem_id"] not in nc_done]
    print(f"No-COT ({task_name}): {len(nc_done)} done, {len(nc_todo)} remaining")

    for i in tqdm(range(0, len(nc_todo), CHUNK_SIZE), desc=f"No-COT ({task_name})"):
        chunk = nc_todo[i:i + CHUNK_SIZE]
        prompts = [
            tokenizer.apply_chat_template(
                build_no_cot_messages(ex["question"]),
                tokenize=False, add_generation_prompt=True
            ) for ex in chunk
        ]
        outputs = llm.generate(prompts, sampling_short)
        for ex, output in zip(chunk, outputs):
            generated = output.outputs[0].text.strip()
            result = {
                "problem_id": ex["problem_id"],
                "condition": nc_cond,
                "model_used": PRIMARY_MODEL,
                "prefill_text": None,
                "predicted_answer": extractor(generated),
                "gold_answer": ex["gold_answer"],
                "generated_tokens": generated,
                "error": None,
            }
            (PREFILL_CACHE / f"{nc_cond}_{ex['problem_id']}.json").write_text(json.dumps(result))

    print(f"{task_name} complete.")

In [ ]:
del llm
gc.collect()
torch.cuda.empty_cache()

# Summary
for task_name in tasks:
    task_cot_dir = CACHE_DIR / f"cots_{task_name}"
    n_cots = len(list(task_cot_dir.glob("*.json")))
    n_nc = len(list(PREFILL_CACHE.glob(f"no_cot_{task_name}_*.json")))
    print(f"{task_name}: {n_cots} COTs, {n_nc} no-COT results")